## 4. Fetch Statistics from Source NetCDF

Replaces the Rasdaman WCS approach. Reads `seg_combined.nc` — the source file used to
populate the Rasdaman coverage — and computes ensemble delta statistics, writing
`stats_diff.csv` with the same 66-column structure.

**Source dims:** `(source: 3, landcover: 2, model: 14, scenario: 5, era: 4, stream_id: 56460)`

**Selection:**
- Projected: `source=2` (gcm_diff_applied_to_maurer) · `landcover=1` (static) · `scenario=3` (rcp60) · `era=2` (2046-2075)
- Historical: `source=0` (original_gcm) · `landcover=1` · `model=8` (Maurer) · `scenario=0` · `era=0` (1976-2005)
- rcp60 models: ACCESS1-0 (0) and BNU-ESM (2) excluded

In [1]:
import numpy as np
import xarray as xr
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [ ]:
NC_PATH        = Path("/beegfs/CMIP6/jdpaul3/hydroviz_data/maurer/nc_stats_fix/seg_combined.nc")
CSV_PATH       = Path("stats_diff.csv")
SHAPEFILE_PATH = "/import/beegfs/CMIP6/jdpaul3/hydroviz_data/gis/xwalk/seg_h8_outlets.shp"

# Dimension integer encodings — coords are float32 in seg_combined.nc
SOURCE_PROJ   = 2.0   # gcm_diff_applied_to_maurer
SOURCE_HIST   = 0.0   # original_gcm
LANDCOVER     = 1.0   # static
MODEL_MAURER  = 8.0
SCENARIO_PROJ = 3.0   # rcp60
SCENARIO_HIST = 0.0   # historical
ERA_PROJ      = 2.0   # 2046-2075
ERA_HIST      = 0.0   # 1976-2005

# Models that participated in rcp60 (ACCESS1-0=0 and BNU-ESM=2 excluded)
RCP60_MODELS = [1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 9.0, 10.0, 11.0, 12.0, 13.0]

In [3]:
VARS_META = {
    "ma12": {"stats": ["ma12_min_d", "ma12_avg_d", "ma12_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma13": {"stats": ["ma13_min_d", "ma13_avg_d", "ma13_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma14": {"stats": ["ma14_min_d", "ma14_avg_d", "ma14_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma15": {"stats": ["ma15_min_d", "ma15_avg_d", "ma15_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma16": {"stats": ["ma16_min_d", "ma16_avg_d", "ma16_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma17": {"stats": ["ma17_min_d", "ma17_avg_d", "ma17_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma18": {"stats": ["ma18_min_d", "ma18_avg_d", "ma18_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma19": {"stats": ["ma19_min_d", "ma19_avg_d", "ma19_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma20": {"stats": ["ma20_min_d", "ma20_avg_d", "ma20_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma21": {"stats": ["ma21_min_d", "ma21_avg_d", "ma21_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma22": {"stats": ["ma22_min_d", "ma22_avg_d", "ma22_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma23": {"stats": ["ma23_min_d", "ma23_avg_d", "ma23_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "dh1":  {"stats": ["dh1_min_d",  "dh1_max_d"],                 "aggs": ["min","max"],       "diff": "pct"},
    "dl1":  {"stats": ["dl1_min_d",  "dl1_max_d"],                 "aggs": ["min","max"],       "diff": "pct"},
    "dh15": {"stats": ["dh15_min_d", "dh15_avg_d", "dh15_max_d"], "aggs": ["min","mean","max"], "diff": "abs"},
    "dl16": {"stats": ["dl16_min_d", "dl16_avg_d", "dl16_max_d"], "aggs": ["min","mean","max"], "diff": "abs"},
    "fh1":  {"stats": ["fh1_min_d",  "fh1_avg_d",  "fh1_max_d"],  "aggs": ["min","mean","max"], "diff": "abs"},
    "fl1":  {"stats": ["fl1_min_d",  "fl1_avg_d",  "fl1_max_d"],  "aggs": ["min","mean","max"], "diff": "abs"},
    "th1":  {"stats": ["th1_min_d",  "th1_avg_d",  "th1_max_d"],  "aggs": ["min","mean","max"], "diff": "circular"},
    "tl1":  {"stats": ["tl1_min_d",  "tl1_avg_d",  "tl1_max_d"],  "aggs": ["min","mean","max"], "diff": "circular"},
    "ma99": {"stats": ["ma99_min_d", "ma99_avg_d", "ma99_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
}

CSV_COLUMNS = (
    [s for v in ["ma12","ma13","ma14","ma15","ma16","ma17","ma18","ma19","ma20","ma21","ma22","ma23"]
     for s in VARS_META[v]["stats"]]
    + ["dh1_min_d","dh1_max_d","dl1_min_d","dl1_max_d"]
    + [s for v in ["dh15","dl16","fh1","fl1","th1","tl1"] for s in VARS_META[v]["stats"]]
    + ["ma99_min_d","ma99_avg_d","ma99_max_d","ma99_hist",
       "dh15_hist","dl16_hist","fh1_hist","fl1_hist"]
)
print(f"CSV_COLUMNS: {len(CSV_COLUMNS)} columns")

CSV_COLUMNS: 66 columns


In [4]:
ds = xr.open_dataset(NC_PATH)
print(f"Loaded {NC_PATH.name}: {dict(ds.sizes)}")

proj = ds.sel(source=SOURCE_PROJ, landcover=LANDCOVER,
              scenario=SCENARIO_PROJ, era=ERA_PROJ)
proj = proj.sel(model=RCP60_MODELS).load()

hist = ds.sel(source=SOURCE_HIST, landcover=LANDCOVER,
              model=MODEL_MAURER, scenario=SCENARIO_HIST, era=ERA_HIST).load()

print(f"Projected: {dict(proj.sizes)}  ({len(proj.model)} rcp60 models)")
print(f"Historical: {dict(hist.sizes)}")

Loaded seg_combined.nc: {'source': 3, 'landcover': 2, 'model': 14, 'scenario': 5, 'era': 4, 'stream_id': 56460}
Projected: {'model': 11, 'stream_id': 56460}  (11 rcp60 models)
Historical: {'stream_id': 56460}


In [5]:
rows = {}
for var, meta in VARS_META.items():
    p = proj[var]   # (model, stream_id)
    h = hist[var]   # (stream_id,)
    if meta["diff"] == "pct":
        denom  = xr.where(h != 0, h, 0.0001)
        deltas = (p - h) / denom * 100
    elif meta["diff"] == "circular":
        deltas = ((p - h + 183) % 366) - 183
    else:
        deltas = p - h
    for stat, agg in zip(meta["stats"], meta["aggs"]):
        rows[stat] = getattr(deltas, agg)(dim="model").round(2).values

for v in ("ma99", "dh15", "dl16", "fh1", "fl1"):
    rows[f"{v}_hist"] = hist[v].round(2).values

df = pd.DataFrame(rows, index=hist.stream_id.values)
df.index.name = "stream_id"
df = df[CSV_COLUMNS]
print(f"Computed: {len(df)} rows x {len(df.columns)} columns")

Computed: 56460 rows x 66 columns


In [6]:
gdf = gpd.read_file(SHAPEFILE_PATH)
shp_ids = set(gdf["seg_id_nat"].tolist())
df = df.loc[df.index.isin(shp_ids)]
print(f"After shapefile filter: {len(df)} rows")

df.to_csv(CSV_PATH)
print(f"Wrote {CSV_PATH}")

After shapefile filter: 56460 rows
Wrote stats_diff.csv


In [7]:
df_check = pd.read_csv(CSV_PATH, index_col="stream_id")
print(f"stats_diff.csv: {len(df_check)} rows, {len(df_check.columns)} columns")
print(f"NaN rows: {df_check.isna().all(axis=1).sum()}")
df_check.head(2)

stats_diff.csv: 56460 rows, 66 columns
NaN rows: 284


,ma12_min_d,ma12_avg_d,ma12_max_d,ma13_min_d,ma13_avg_d,ma13_max_d,ma14_min_d,ma14_avg_d,ma14_max_d,ma15_min_d,...,tl1_avg_d,tl1_max_d,ma99_min_d,ma99_avg_d,ma99_max_d,ma99_hist,dh15_hist,dl16_hist,fh1_hist,fl1_hist
stream_id,,,,,,,,,,,,,,,,,,,,,
1,40.38,139.22,259.13,38.34,220.09,544.05,39.45,296.44,720.44,-44.13,...,-18.88,-4.24,-9.83,-3.24,15.45,117.35,5.42,10.50,17.03,8.30
2,29.07,65.35,109.38,41.12,100.02,224.23,23.71,65.25,100.20,-53.30,...,-4.22,3.92,-3.45,3.51,13.11,374.31,10.04,8.54,8.63,10.37
